# 📐 Module 3 — Class 2: Make Numbers Comparable (Scaling) — Olist Edition

**Lesson:** [bepro-aiml.github.io/aiml-platform/#/module/3/class/2](https://bepro-aiml.github.io/aiml-platform/#/module/3/class/2)

---

## 📖 Today's story

Yesterday you **cleaned** the Olist orders. ✅

Today the boss asks: *"OK, predict which orders will be late!"*

But there is one more problem hiding in the data...

Look at our orders. We have:
- `distance_km` → numbers from **0 to ~5,000** (kilometres customer ↔ seller)
- `total_freight` → numbers from **0 to ~400** (Brazilian reais)
- `total_price` → numbers from **0 to ~7,000** (Brazilian reais)

All are numbers. **But the sizes are very different!**

The model will think: *"distance_km and total_price look most important because they're the biggest!"* ❌

**That is wrong.** A 100 km distance change is NOT less important than a 100 reais price change. They just use different scales.

**Today we make all numbers fair to compare.**

---

## 💡 What is "scaling"?

**Scaling = making different numbers fit in a similar range.**

Example:
- BEFORE: distance = 1500 km, freight = 25 reais (very different sizes)
- AFTER:  distance = 0.5, freight = 0.5 (now they are equal in size)

We do not change WHAT the data means. We only change the **size of the numbers**.

---

## 🎯 Today's plan

1. **Setup** — get the Olist orders we cleaned in Class 1
2. **See the problem** — why different sizes break the model
3. **Tool 1: StandardScaler** — make mean = 0, spread = 1
4. **Tool 2: MinMaxScaler** — squeeze to [0, 1]
5. **Tool 3: RobustScaler** — best when outliers exist
6. **The BIG rule** — fit on train data ONLY (or you cheat!)
7. **Compare** — which scaler wins on a real model predicting `is_late`?
8. **Save** scaled data for Class 3

## 🤖 How to run

Click a code cell → press **Shift + Enter**.

👉 Each cell prints **what to look at** AND **why it matters**. Read the output text!

---

# 🧰 First — What is scikit-learn?

Today is the **first time** we use a new tool: **scikit-learn** (people say *"sklearn"*).

Before we use it, let's understand what it is. This will take ~20 minutes but it will make EVERY future class easier.

## 📦 The toolbox analogy

Imagine a **carpenter**. They have a toolbox:
- 🔨 hammer — for nails
- 🪛 screwdriver — for screws
- 📏 ruler — for measuring
- 🪚 saw — for cutting

Each tool does ONE job. Carpenter picks the right tool for the right job.

**scikit-learn is the same thing — but for machine learning.**

It's a toolbox with hundreds of tools:
- 📐 tools to **prepare data** (scaling, cleaning numbers)
- 🤖 tools to **train models** (Logistic Regression, Random Forest, KNN)
- 📊 tools to **check models** (accuracy, precision, F1)
- 🎯 tools to **find best settings** (auto-tuning)

We pick the right tool for our job.

## 💡 Why do we need sklearn? Why not write code ourselves?

If we wrote everything ourselves:
- ❌ Hundreds of lines of math for ONE model
- ❌ Slow (Python is not fast for math)
- ❌ Bugs! Math errors are hard to find

With sklearn:
- ✅ 2-3 lines per model
- ✅ Very fast (the heavy math is written in low-level C under the hood, so it runs at top speed — you don't have to think about it)
- ✅ Tested by millions of people — almost no bugs

**Almost EVERY ML project in the world uses sklearn. It is the standard.**

## 🌍 Real-world fact

If you go to LinkedIn or a job interview for an ML role:
- They will ask: *"Do you know sklearn?"*
- Not asking: *"Can you write Logistic Regression from scratch?"*

So learning sklearn = learning the job skill.

---

## 🎯 The 4 jobs sklearn does

Every ML project follows the same 4 steps. sklearn has tools for all of them.

| # | Job | What it means | Example tool |
| --- | --- | --- | --- |
| 1 | **Prepare data** | Make data ready for the model | `StandardScaler` |
| 2 | **Split data** | Train + test parts | `train_test_split` |
| 3 | **Train a model** | Teach the model to predict | `LogisticRegression` |
| 4 | **Check the model** | Measure how good it is | `accuracy_score` |

We will use ALL 4 today!

## 💡 The MAGIC of sklearn — every tool works the same way

The carpenter needs to learn each tool separately (hammer is different from saw).

**But sklearn is different.** Every tool follows ONE of two simple patterns. Once you learn the pattern, ALL tools become easy.

Let me explain the 2 patterns 👇

---

## 🛠️ Pattern 1: Preprocessor tools (`fit` then `transform`)

These are tools that **prepare** the data. Like StandardScaler, MinMaxScaler, OneHotEncoder.

They follow 2 steps:

### Step 1: `fit()` — LEARN from the data

The tool looks at the data and **memorizes important numbers**.

For StandardScaler, it memorizes:
- The **mean** (average)
- The **std** (spread)

It does NOT change the data yet. Just learns.

```python
scaler = StandardScaler()
scaler.fit(X_train)   # ← learns mean and std from X_train
```

### Step 2: `transform()` — APPLY what was learned

Now use the memorized numbers to actually change the data.

```python
X_scaled = scaler.transform(X_train)   # ← uses the memorized mean/std to scale
```

### 🎁 Shortcut: `fit_transform()` does BOTH at once

```python
X_scaled = scaler.fit_transform(X_train)   # ← fit AND transform in one line
```

**WARNING:** Use `fit_transform` ONLY on training data. Use `transform` (no fit) on test data!

### 🍞 Real-life analogy

Imagine making **bread**:
- `fit()` = read the recipe (learn the rules)
- `transform()` = make the bread (apply the rules)
- `fit_transform()` = read the recipe and make the bread together

You read the recipe ONCE. Then you can make many breads. Same with sklearn — `fit` once on training data, then `transform` many times on new data.

---

## 🤖 Pattern 2: Model tools (`fit` then `predict` then `score`)

These are tools that **predict** something. Like Logistic Regression, Random Forest, KNN.

They follow 3 steps:

### Step 1: `fit()` — LEARN from training data

```python
model = LogisticRegression()
model.fit(X_train, y_train)   # ← learns from features + correct answers
```

Notice: this `fit` takes **two** things:
- `X_train` = the features (numbers about each order)
- `y_train` = the correct answer for each order (was it late? Yes/No)

### Step 2: `predict()` — GUESS the answer for new data

```python
predictions = model.predict(X_test)   # ← guesses Yes/No for each test order
```

### Step 3: `score()` — CHECK how often the model is right

```python
accuracy = model.score(X_test, y_test)
```

### 🎓 Real-life analogy

Imagine teaching a child:
- `fit()` = show them many examples ("This Olist order arrived late. This one arrived on time.")
- `predict()` = ask them about new orders ("Will THIS order be late?")
- `score()` = check how many they got right (8 out of 10 = 80%)

**That's it!** Two patterns. Every sklearn tool uses one of them.

---

## 🎬 Hello sklearn — let's actually USE it

Before we touch our Olist data, let's run a **tiny example** so you SEE the patterns work.

The next cell:
1. Takes 5 simple numbers: `[10, 20, 30, 40, 50]`
2. Uses StandardScaler to scale them
3. Shows what changed

This is the simplest possible sklearn example. After this, the rest of the lesson will feel easy.

In [ ]:
# 🎬 HELLO SKLEARN — tiniest possible example

# Step 1: import the tool
from sklearn.preprocessing import StandardScaler

# Step 2: get our tiny data — 5 simple numbers
# (Note the [[ ]] — sklearn always wants a 2D table, not 1D)
data = [[10], [20], [30], [40], [50]]

print('🔢 BEFORE scaling:')
print(f'   Numbers: {[row[0] for row in data]}')
print(f'   Mean   : 30.0')
print(f'   Range  : 10 to 50')
print()

# Step 3: create the tool
scaler = StandardScaler()

# Step 4: fit + transform (the magic 1-liner!)
scaled = scaler.fit_transform(data)

print('✅ AFTER scaling:')
print(f'   Numbers: {[round(row[0], 2) for row in scaled]}')
print(f'   Mean   : {scaled.mean():.2f}  ← became 0!')
print(f'   Std    : {scaled.std():.2f}  ← became 1!')
print()
print('━' * 50)
print('💡 What happened?')
print('   • fit() learned: mean=30, std=14.14')
print('   • transform() applied: new = (old - 30) / 14.14')
print('   • Result: mean=0, std=1 (the StandardScaler magic!)')
print('━' * 50)

---

## 📦 What we'll import today (and why)

Now you know what sklearn is. Below are the tools we'll import for today's lesson:

| Import | What it does | Pattern |
| --- | --- | --- |
| `StandardScaler` | Scale: mean=0, std=1 | Preprocessor (fit + transform) |
| `MinMaxScaler` | Scale: range [0, 1] | Preprocessor (fit + transform) |
| `RobustScaler` | Scale: uses median + IQR | Preprocessor (fit + transform) |
| `train_test_split` | Split data into 2 parts | Function (not a tool to fit) |
| `LogisticRegression` | Predict yes/no on the `is_late` target | Model (fit + predict + score) |
| `accuracy_score` | Measure: % of correct guesses | Function (compare 2 lists) |

You'll see all 6 today. **You already used StandardScaler above** — congratulations, you're using sklearn!

🎯 **Now we're ready.** Let's start the real lesson with our Olist data 👇

---
## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

# We will use sklearn — the most popular ML library in Python.
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print('✅ Tools ready! We are using:')
print('   • pandas, numpy, matplotlib (you know these from Module 2)')
print('   • sklearn = scikit-learn (the BIG ML library)')

---

## 📖 Quick reference — words and code idioms you'll see today

Before we dive in, here's a tiny reference card. Don't memorize anything. Just **scroll back here** when something looks weird in the code below.

### 🧠 5 ML words you'll see everywhere

| Word | What it means | Example in our Olist data |
| --- | --- | --- |
| **feature** | An input column the model uses to predict something | `distance_km`, `total_freight`, `total_price` |
| **target** | The column we want to predict (the answer) | `is_late` (was the order delivered late? 1 = yes, 0 = no) |
| **X** (uppercase) | All features together → a 2D **table** | `df[['distance_km','total_freight','total_price']]` |
| **y** (lowercase) | The target column → a 1D **list** | `df['is_late']` |
| **production** | The model running for real on real new orders (after class!) | Your code on Olist's website |

> 💡 **Memory trick:** *"X is what we **know** about an order. y is what we **want to know**."*
>
> Example: I know an order's distance, freight, price (X). I want to know: will it be late? (y).

### 🐍 Python tricks you'll see in code below

| What you'll see | What it does (in plain English) |
| --- | --- |
| `pd.read_parquet('orders_step1.parquet')` | Load the file Class 1 saved |
| `.astype(int)` | Convert column to whole numbers (0, 1, 2 ...) |
| `.describe()` | Pandas: show summary stats (mean, std, min, max) for every column |
| `random_state=42` | Lock the randomness — you get the **SAME split** every run, so results are reproducible |
| `joblib.dump(thing, 'file')` | Save any Python object to disk so you can re-use it tomorrow |
| `warnings.filterwarnings('ignore')` | Hide ugly orange warning messages — keeps the notebook clean |
| `[[10],[20],[30]]` (double brackets) | A 2D list. **sklearn always wants 2D**, never 1D — that's why you see `[[ ]]` a lot |

🎯 You don't need to remember all of these now. They will appear again — and again — and you'll get used to them naturally. This list is just a **safety net** for when you get stuck.

---
## Step 1: Get our Olist data 📥

We need the file we cleaned in Class 1 (`orders_step1.parquet`).

If you don't have it yet, you must finish the Class 1 homework first.

In [ ]:
# Load the Class 1 output
df = pd.read_parquet('orders_step1.parquet')

# We also need to merge in items + payments to get total_price, total_freight, distance_km.
# (This is also what Class 1 Gold homework asked you to start.)
items = pd.read_csv('olist_data/olist_order_items_dataset.csv')

# Aggregate to per-order totals (one row per order_id)
order_totals = items.groupby('order_id').agg(
    num_items=('order_item_id', 'count'),
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
).reset_index()

df = df.merge(order_totals, on='order_id', how='left')

# For today's scaling demo we need the engineered distance_km too.
# We'll skip the heavy geo merge — instead, simulate distance_km from a simple proxy.
# (In Class 3 you'll do the proper Haversine. Today we just need 3 columns with different scales.)
rng = np.random.default_rng(42)
df['distance_km'] = (df['total_freight'].fillna(20).clip(0, 50) * 30 +
                     rng.uniform(50, 500, size=len(df)))

# Engineer is_late (we'll use it in step 8)
df['is_late'] = ((df['order_delivered_customer_date'] >
                  df['order_estimated_delivery_date']).astype(int))

print(f'✅ Loaded {len(df):,} Olist orders. Ready to scale!')
df[['distance_km','total_freight','total_price','is_late']].head()

---
## Step 2: See the problem 👀

> **Why do we need scaling?**

Let's look at 3 number columns and see how different their sizes are.

In [ ]:
cols = ['distance_km', 'total_freight', 'total_price']

print('🔍 Range of values for each column:')
print()
for c in cols:
    print(f'  {c:18s} : min = {df[c].min():>9.1f}  max = {df[c].max():>9.1f}')
print()
print('━' * 55)
print('🚨 PROBLEM: The 3 columns are on VERY different scales!')
print()
print('   distance_km    goes from 0 to ~5,000   (BIG numbers)')
print('   total_freight  goes from 0 to ~400     (small numbers)')
print('   total_price    goes from 0 to ~7,000   (BIG numbers)')
print()
print('💡 The model will think distance_km and total_price')
print('   are the "most important" features — just because')
print('   the numbers are bigger. This is WRONG.')
print('━' * 55)

### 🤔 Why does this happen?

Many ML models use **distance** to compare orders.

**Distance** = how far apart 2 orders are. Like measuring 2 cities on a map.

Imagine 2 orders:
- Order A: distance = 100 km, freight = 20, price = 50
- Order B: distance = 105 km, freight = 22, price = 60

Difference:
- distance: 5 (small)
- freight: 2 (small)
- **price: 10 (LOOKS bigger!)**

The model sees: *"Order A and Order B are very different — price differs by 10!"*

But really, all 3 differences mean the same thing! Price just LOOKS bigger because of the scale.

👉 **Scaling fixes this.** It makes a 5 km distance difference equally important to a 10-real price difference.

### Which models care about scaling?

| Model type | Cares? | Why |
| --- | --- | --- |
| 📏 **K-NN, K-Means** | ✅ YES | They use distance |
| 📈 **Linear / Logistic Regression** | ✅ YES | Math works better with similar sizes |
| 🧠 **Neural networks** | ✅ YES | Same — math works better |
| 🌳 **Decision Trees, Random Forest** | ❌ No | They split by 'is X > 5?' — size doesn't matter |

💡 **Pro tip:** When in doubt, scale. It rarely hurts trees, and it really helps everything else.

---
## Step 3: Tool 1 — StandardScaler 📊

### What is it?

**StandardScaler** = the most popular scaler. It changes each number so:
- The **mean** (average) becomes **0**
- The **spread** (standard deviation) becomes **1**

### How does it work? (the simple math)

For each value: `new_value = (old_value - mean) / standard_deviation`

**In words:** *"How many standard deviations is this value from the average?"*

Don't worry about the math — sklearn does it for us!

### When to use?

✅ **Default choice** — use it when you don't know what to do.
✅ Good for most models (Logistic Regression, KNN, Neural Networks).
❌ Bad when there are huge outliers (use RobustScaler instead — see Step 5).

---

### ⚠️ Hold on! What about NEGATIVE numbers after scaling?

When you run StandardScaler, you'll see numbers like `-1.32`, `-0.91`, `+1.61`. 

**What do these mean? Let's figure it out together with REAL math.**

### 🧮 The formula in 3 simple words

```
new_value = (old_value - mean) / standard_deviation
```

In plain English:
> *"How many **steps** away is this value from the average?"*

Where each "step" = one **standard deviation** (the typical spread).

### What the sign tells us

| Sign | Meaning |
| --- | --- |
| **POSITIVE** (e.g. +1.61) | Value is **ABOVE** the average |
| **ZERO** (0.00) | Value is **EQUAL** to the average |
| **NEGATIVE** (e.g. -1.32) | Value is **BELOW** the average |

### What the size tells us

| Size | Meaning |
| --- | --- |
| Between -1 and +1 | **Normal** order (close to average) |
| Beyond ±2 | **Unusual** order (far from average) |
| Beyond ±3 | **Very rare** order (might be outlier!) |

### 🎯 Think of it like this

An order with `distance_km = 4500` becomes `+2.20`. This means:
> *"This order's distance is **2.2 steps ABOVE** the average distance."*

An order with `distance_km = 50` becomes `-1.10`. This means:
> *"This order's distance is **1.1 steps BELOW** the average distance."*

The numbers are not random! Let's calculate them BY HAND in the next cell.

### 🤔 Wait — what is "mean" and "std" again?

Before we believe the math, let's make sure we know these 2 simple words.

**Mean** = the average of all values.
- Example: ages of 5 students: [10, 12, 14, 16, 18]
- Mean = (10+12+14+16+18) / 5 = **14**

**Standard deviation (std)** = how SPREAD OUT the values are from the mean.
- Small std → values are CLOSE to the mean (like [13, 14, 14, 14, 15] → small std)
- Big std → values are FAR from the mean (like [5, 10, 14, 18, 23] → big std)
- It's the "typical distance" from the mean

**For our `distance_km` column in Olist:**
- mean ≈ 1100 km → average order ships ~1100 km
- std ≈ 1500 km → typical order is ~1500 km from average (some go 200 km, some go 4500 km — that's the typical spread)

OK — now let's run the math for real Olist orders 👇

In [ ]:
# 🧮 Let's do the math BY HAND on real orders from our Olist data

# Step 1: Find the mean and std of the 'distance_km' column
mean_d = df['distance_km'].mean()
std_d  = df['distance_km'].std()

print('🔢 For the distance_km column in our Olist data:')
print(f'   mean = {mean_d:.2f}  ← average distance across all orders')
print(f'   std  = {std_d:.2f}  ← typical spread (one "step")')
print()
print('━' * 60)
print('📐 The formula:  new = (old - mean) / std')
print('━' * 60)
print()

# Step 2: Pick 4 example distance values and walk through the math
examples = [
    (4800, 'A LONG-DISTANCE order (Manaus → São Paulo)'),
    (1100, 'An AVERAGE order'),
    (250,  'A SHORT order (within same state)'),
    (50,   'A LOCAL order (same city)')
]

for val, label in examples:
    new_val = (val - mean_d) / std_d
    sign_word = 'ABOVE' if new_val > 0 else ('BELOW' if new_val < 0 else 'EQUAL TO')
    print(f'📦 {label}: distance = {val} km')
    print(f'   Step 1: subtract the mean')
    print(f'           {val} - {mean_d:.2f} = {val - mean_d:+.2f}')
    print(f'   Step 2: divide by the std')
    print(f'           {val - mean_d:+.2f} / {std_d:.2f} = {new_val:+.2f}')
    print(f'   ✨ Result: {new_val:+.2f}  ({sign_word} the average)')
    print()

print('━' * 60)
print('💡 NOW you understand!')
print('   • Negative numbers = order is BELOW average distance')
print('   • Positive numbers = order is ABOVE average distance')
print('   • Zero = order is EXACTLY the average')
print('   • The size = how many "steps" away')
print('━' * 60)

---

## 🎯 So why does this matter for the model?

You just saw the math. Now let's connect it back to **why** we did all this.

### Before scaling — 3 columns speaking 3 different languages

| Column | Range | Typical value |
| --- | --- | --- |
| distance_km | 0 to 5,000 | **BIG numbers** |
| total_freight | 0 to 400 | small numbers |
| total_price | 0 to 7,000 | **BIG numbers** |

The model thinks: *"distance_km and total_price must be more important — their numbers are bigger!"* ❌

### After StandardScaler — every column speaks the SAME language

| Column | New range | Typical value |
| --- | --- | --- |
| distance_km | -1.10 to +2.20 | between -1 and +1 |
| total_freight | -1.00 to +5.00 | between -1 and +1 |
| total_price | -0.40 to +3.50 | between -1 and +1 |

The model now thinks: *"All 3 columns are equally important. I will compare them fairly."* ✅

### 🍎 The big idea (kid-level summary)

Imagine 3 kids playing a game:
- 🏃 Ali shouts numbers (loud!)
- 🗣️ Zara talks normally
- 🤫 Jasur whispers (quiet!)

If you only listen to volume, **Ali always wins**. Not fair!

Scaling = giving everyone the same microphone. Now everyone is equally loud.

### 🎓 Three things to remember

1. **Negative ≠ Bad.** Negative just means "below the average." It's a normal thing.
2. **Zero = average order.** A scaled value of 0 means *exactly average*.
3. **Size = how unusual.** Bigger numbers (further from 0) = more unusual order.

Now we are ready to apply StandardScaler to all 3 columns at once 👇

In [ ]:
# Apply StandardScaler to our 3 columns
scaler = StandardScaler()              # 1. Create the tool
X = df[cols].fillna(0)                  # 2. Pick the columns (fillna for safety)
X_standard = scaler.fit_transform(X)    # 3. Fit + transform = learn + apply

# Put back in a DataFrame for nicer printing
X_standard_df = pd.DataFrame(X_standard, columns=cols)

print('✅ StandardScaler applied!')
print()
print('🔍 BEFORE (original values):')
print(X.describe().loc[['mean', 'std', 'min', 'max']].round(2))
print()
print('🔍 AFTER StandardScaler:')
print(X_standard_df.describe().loc[['mean', 'std', 'min', 'max']].round(2))
print()
print('━' * 55)
print('👉 What changed:')
print('   • mean is now ~0 (was big and different per column)')
print('   • std is now ~1')
print('   • All 3 columns are now on the SAME scale!')
print('━' * 55)

---
## Step 4: Tool 2 — MinMaxScaler 📐

### What is it?

**MinMaxScaler** = squeezes all values into a fixed range, usually **[0, 1]**.

- The smallest value → becomes 0
- The biggest value → becomes 1
- Everything else → spread between 0 and 1

### Why use it?

Some models work BEST with inputs in [0, 1]. Examples:
- 🧠 **Neural networks** — small inputs (close to 0–1) help the neurons learn smoothly. Big inputs make them confused.
- 🖼️ **Image processing** — every pixel in a photo is already 0 to 255. Mapping that to 0 to 1 is the natural fit.

### How it works

`new_value = (old_value - min) / (max - min)`

In plain words: *"how far between the smallest and the biggest is this value?"* — 0 means smallest, 1 means biggest.

### When to use?

✅ When the model needs values in [0, 1]
✅ When data has NO big outliers
❌ **Dangerous with outliers** — see what happens below!

In [ ]:
# Apply MinMaxScaler
scaler = MinMaxScaler()
X_minmax = scaler.fit_transform(X)
X_minmax_df = pd.DataFrame(X_minmax, columns=cols)

print('✅ MinMaxScaler applied!')
print()
print('🔍 AFTER MinMaxScaler:')
print(X_minmax_df.describe().loc[['min', 'max', 'mean']].round(3))
print()
print('━' * 55)
print('👉 Notice: min = 0.0 and max = 1.0 for ALL columns!')
print('   Everything is now squeezed into [0, 1].')
print('━' * 55)

---

## 🛑 Wait — what is an "outlier"?

You're about to read the word **outlier** a lot. Let's nail down what it means BEFORE we go further.

### Outlier = a number that is FAR away from the rest

Imagine you measure the freight of 10 Olist orders:

| Order | Freight (R$) |
| --- | --- |
| 1 | 15 |
| 2 | 22 |
| 3 | 18 |
| 4 | 30 |
| 5 | 25 |
| 6 | 19 |
| 7 | 35 |
| 8 | 28 |
| 9 | 21 |
| 10 | **🚨 9999** ← OUTLIER |

Order 10 has 9,999 reais of freight. That's 250× the typical freight. **Very strange.**

It's probably:
- A **typo** (someone wrote 9999 instead of 99)
- A **bug** (system glitch put 9999 instead of the real value)
- A **fraud** signal (someone tested with a huge value)
- Or one truly rare strange case (a refrigerator shipped to the Amazon by air)

### Picture in your head

```
the crowd                                   the loner
●●●●●●●●●                                       🚨
15  20  25 ............................. 9999
←——— most data here ———→                  ←— far away —
```

The outlier is the lonely dot far from the crowd.

### Where outliers come from in Olist data

- A freight of 999 R$ on a 50 R$ order (system bug)
- A `total_price` of 7000 R$ for a single product (real but rare — maybe a TV)
- A `delivery_days` of 200 (real — package lost and re-routed through 3 cities)
- A duplicate seller_id with weird zip code (data entry error)

Some are mistakes. Some are real-but-rare. Either way → they make the math go crazy.

### Why does this matter NOW?

Outliers can **completely break** some scaling tools and **barely affect** others. In the next cells we'll see exactly how each scaler reacts to outliers.

The first victim is **MinMaxScaler** 👇

### 🚨 The MinMax outlier trap

Imagine ONE order has `total_freight = 99,999` (typo or fraud). What happens?

MinMax sees: max = 99,999. So it makes 99,999 → 1.0.

But everyone else (paying R$10–R$50) gets squeezed into a TINY space at the bottom — like 0.0001 to 0.0005!

**Result:** 99% of your data uses only 0.05% of the [0, 1] range. Most info is lost.

**Lesson:** Never use MinMaxScaler if you have wild outliers. Clean outliers FIRST, or use RobustScaler instead.

In [ ]:
# DEMO: what happens to MinMax if we add ONE outlier?
X_with_outlier = X.copy()
X_with_outlier.iloc[0, X_with_outlier.columns.get_loc('total_freight')] = 99999  # one bad row

scaler = MinMaxScaler()
X_outlier_scaled = scaler.fit_transform(X_with_outlier)
outlier_df = pd.DataFrame(X_outlier_scaled, columns=cols)

print('🚨 DEMO: What ONE outlier does to MinMax')
print()
print('total_freight values after MinMax (with one 99999 outlier):')
print(f'   The outlier row    : {outlier_df.iloc[0]["total_freight"]:.4f}')
print(f'   95% of orders are ≤: {outlier_df["total_freight"].quantile(0.95):.4f}')
print(f'   50% of orders are ≤: {outlier_df["total_freight"].quantile(0.5):.4f}')
print()
print('━' * 55)
print('👉 99,999 → 1.0 (the max)')
print('   95% of REAL orders got squeezed into [0, 0.005]')
print('   That is only 0.5% of the [0,1] space!')
print('   Most of our information is LOST.')
print('━' * 55)

---

## 🧪 OK, MinMax dies with one outlier. What about the other scalers?

You just saw what ONE outlier did to MinMax — it ate **99% of our information**.

But what about the OTHER tools?

- 🤔 Does an outlier hurt **StandardScaler** too? (it uses **mean** and **std**)
- 🤔 Why is **RobustScaler** supposed to be **safe**? (it uses **median** and **IQR**)

Let's not just trust me — let's **measure it ourselves** in the next cell. We'll take 10 normal freight values, add ONE crazy outlier (`9999`), then watch what happens to all 4 stats: mean, std, median, IQR.

This is the cell that proves **why RobustScaler exists**.

In [ ]:
# 🧪 Watch what an outlier does to mean/std vs median/IQR

# 10 normal Olist freight values + 1 crazy outlier
normal   = [15, 18, 22, 25, 28, 30, 33, 38, 42, 50]
with_bad = normal + [9999]     # add ONE outlier

q1_n, q3_n = np.percentile(normal,   [25, 75])
q1_b, q3_b = np.percentile(with_bad, [25, 75])

print('🔍 WITHOUT outlier (10 normal Olist orders):')
print(f'   mean   = {np.mean(normal):>7.1f}')
print(f'   std    = {np.std(normal):>7.1f}')
print(f'   median = {np.median(normal):>7.1f}')
print(f'   IQR    = {q3_n - q1_n:>7.1f}')
print()
print('💥 WITH ONE outlier (10 normal + 9999):')
print(f'   mean   = {np.mean(with_bad):>7.1f}    ← went CRAZY (~30x bigger!)')
print(f'   std    = {np.std(with_bad):>7.1f}    ← went CRAZY (~250x bigger!)')
print(f'   median = {np.median(with_bad):>7.1f}    ← barely moved 👍')
print(f'   IQR    = {q3_b - q1_b:>7.1f}    ← barely moved 👍')
print()
print('━' * 60)
print('💡 LESSON:')
print('   • mean and std are EASILY ruined by outliers')
print('   • median and IQR barely change — they are STRONG')
print()
print('   → StandardScaler uses mean/std → gets ruined by outliers')
print('   → RobustScaler uses median/IQR → stays safe!')
print('━' * 60)

---

## 📏 What is "median" and "IQR"?

These 2 simple words = the **secret** behind RobustScaler.

### Median = the MIDDLE value (when data is sorted)

Take 7 freight values, sort them, pick the middle one:

```
[15, 20, 22, 28, 35, 42, 50]
             ↑
          median = 28
```

Now change the highest value from 50 to 9999:

```
[15, 20, 22, 28, 35, 42, 9999]
             ↑
          median is STILL 28  ← outlier did nothing!
```

**Outliers cannot move the median.** ✊ That's the magic.

### Quartiles — split sorted data into 4 equal groups

Imagine sorting 100 Olist orders from cheapest freight to most expensive. Cut them into 4 equal groups of 25 each:

```
[smallest 25%] | [next 25%] | [next 25%] | [biggest 25%]
                ↑            ↑            ↑
                Q1        median (Q2)     Q3
              (25% mark)   (50% mark)   (75% mark)
```

- **Q1** (first quartile) = the value at the **25%** mark
- **Q2** = the **median** (50% mark)
- **Q3** (third quartile) = the value at the **75%** mark

### IQR = Q3 − Q1 = the middle 50% range

**IQR** stands for **Inter-Quartile Range**. It tells us:

> *"How spread out is the **middle half** of the data?"*

**Why is this beautiful?** We **ignore** the smallest 25% and biggest 25%. So any outliers (which sit at the edges) are **cut off** and can't break the math.

### Real Olist example

Freight values: `[10, 15, 20, 25, 30, 35, 40, 50, 9999]`

- **Median** = 30 (middle value, ignores the 9999)
- **Q1** = ~17 (one quarter from bottom)
- **Q3** = ~45 (three quarters from bottom)
- **IQR** = 45 − 17 = **28** ← the middle half is 28 wide

Even with 9999 in there, IQR is just 28. **Outlier-proof!** 🛡️

Now you understand what RobustScaler is doing under the hood: it uses *(value − median) / IQR* instead of *(value − mean) / std*. Same idea, but with outlier-proof tools.

---
## Step 5: Tool 3 — RobustScaler 🛡️

### What is it?

**RobustScaler** = a scaler that is **robust** (= strong, not affected) to outliers.

Instead of using mean and std (which outliers ruin), it uses:
- The **median** (middle value)
- The **IQR** (the middle 50% range — Q3 minus Q1)

### Why it works

The median doesn't change if there's an outlier. The IQR doesn't change either. So a few crazy values can NOT mess up the scaling.

### When to use?

✅ When data has outliers you can't easily remove
✅ Skewed distributions (values lean to one side)
✅ E-commerce / banking / fraud data — always has outliers

In [ ]:
# Apply RobustScaler to the data WITH the outlier
scaler = RobustScaler()
X_robust = scaler.fit_transform(X_with_outlier)
robust_df = pd.DataFrame(X_robust, columns=cols)

print('🛡️ RobustScaler applied (to data WITH the 99999 outlier):')
print()
print('total_freight values after RobustScaler:')
print(f'   The outlier row     : {robust_df.iloc[0]["total_freight"]:.2f}')
print(f'   95% of orders are ≤ : {robust_df["total_freight"].quantile(0.95):.2f}')
print(f'   50% of orders are ≤ : {robust_df["total_freight"].quantile(0.5):.2f}')
print()
print('━' * 55)
print('👉 The outlier is now still big (correct!)')
print('   But REAL orders get a NORMAL spread (around 0).')
print('   No information lost!')
print('   This is why RobustScaler wins when outliers exist.')
print('━' * 55)

---
## Step 6: Compare all 3 scalers side by side 📊

In [ ]:
# Visual comparison: histogram of total_freight with each scaler
fig, ax = plt.subplots(1, 4, figsize=(16, 3.5))

data_views = [
    ('Original',       X['total_freight']),
    ('StandardScaler', X_standard_df['total_freight']),
    ('MinMaxScaler',   X_minmax_df['total_freight']),
    ('RobustScaler',   pd.DataFrame(RobustScaler().fit_transform(X), columns=cols)['total_freight'])
]

for i, (name, data) in enumerate(data_views):
    ax[i].hist(data, bins=40, color='steelblue', edgecolor='white')
    ax[i].set_title(name)
    ax[i].set_xlabel('value')
    if i == 0: ax[i].set_ylabel('count')

plt.suptitle('total_freight (Olist) — same data, 4 different scales', fontsize=14)
plt.tight_layout()
plt.show()

print()
print('━' * 55)
print('👉 What you see in each chart:')
print()
print('  1. Original       → x-axis: 0 to ~400 (real reais)')
print('  2. StandardScaler → x-axis: ~-1 to ~5 (mean=0, std=1, but outliers stretch right tail)')
print('  3. MinMaxScaler   → x-axis: 0 to 1   (squeezed)')
print('  4. RobustScaler   → x-axis: ~-1 to ~3 (centered on median)')
print()
print('💡 The SHAPE is the same — only the SIZE changes!')
print('   Scaling does not change the data. It changes the scale.')
print('━' * 55)

---

## 🎁 Bonus — are these the ONLY scalers?

We learned **3** today: StandardScaler, MinMaxScaler, RobustScaler.

These are the **3 most popular**. But sklearn has more! Here's a fuller picture:

| Scaler | What it does | When to use |
| --- | --- | --- |
| **StandardScaler** ⭐ | mean=0, std=1 | **Default. Use this 80% of the time.** |
| **MinMaxScaler** | squeeze to [0, 1] | When the model needs values in [0,1] (neural nets, images) |
| **RobustScaler** 🛡️ | uses median + IQR | When data has wild outliers |
| **MaxAbsScaler** | divide by biggest absolute value | When data is sparse (lots of zeros) |
| **Normalizer** | makes each ROW sum to 1 | Text/document data — note: works on rows, not columns |
| **PowerTransformer** | makes data look like a bell curve | When data is very skewed |
| **QuantileTransformer** | maps to a normal distribution | Strong fix for weird-shaped data |

### 💡 You don't need to memorize all 7

**The simple rule of thumb:**

1. **First try:** `StandardScaler` ← works for ~80% of cases
2. If outliers exist → switch to `RobustScaler`
3. If model needs [0, 1] → switch to `MinMaxScaler`
4. The other 4? You'll meet them later when you actually need them.

### 🌍 In real ML jobs

- **StandardScaler** is the most common. **Memorize this one first.**
- **MinMaxScaler** for neural networks / image data (pixel values 0-255 → 0-1).
- **RobustScaler** for finance / banking / e-commerce — these fields ALWAYS have outliers (Olist included).

These 3 cover 99% of real work. ✅

---
## Step 7: 🚨 The BIG RULE — fit on TRAIN data only

This is the **most important rule** of the whole module. Listen carefully.

### What is train/test split?

Before training a model, we split the data into 2 parts:
- **Train data** (~80%) — the model learns from this
- **Test data** (~20%) — we keep it secret to check the model later

Like an exam:
- Train data = practice questions
- Test data = the real exam (model has never seen it)

### What is data leakage?

**Data leakage = information from test data secretly leaks into training.**

Imagine you are a teacher. You give Ali the exam answers BEFORE the exam. Ali gets 100%. But Ali doesn't know math!

Same with our model. If the scaler "sees" the test data, the model is **cheating**. It looks great on the test but fails in real life (production).

### The wrong way ❌

```python
# WRONG! Scaler sees ALL data including test
X_scaled = scaler.fit_transform(X)
X_train, X_test = train_test_split(X_scaled)
```

### The right way ✅

```python
# Split FIRST
X_train, X_test = train_test_split(X)

# Scaler learns from train ONLY
X_train_scaled = scaler.fit_transform(X_train)

# Same scaler is applied to test (no learning, just apply)
X_test_scaled = scaler.transform(X_test)
```

### Why this matters in production

In real life, when a NEW Olist order arrives, you don't know its data yet. You have to scale it using the **old** scaler from training. So that's exactly how we should test the model — apply the train scaler to test data.

**Memorize:** *fit_transform on TRAIN, transform on TEST. Never both.*

In [ ]:
# Show the right way in code
y = df['is_late']  # target column (1 = late, 0 = on time)

# STEP 1: split FIRST
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

# STEP 2: fit scaler on train ONLY
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# STEP 3: apply same scaler to test (no fit!)
X_test_scaled = scaler.transform(X_test)

print('✅ The right way!')
print(f'   Train rows: {len(X_train):,}')
print(f'   Test rows : {len(X_test):,}')
print()
print('   Train distance_km:')
print(f'     Before scaling: mean = {X_train["distance_km"].mean():.2f}')
print(f'     After scaling : mean = {X_train_scaled[:, 0].mean():.2f}  (should be ~0)')
print()
print('💡 The scaler ONLY saw training data when learning.')
print('   Test data was scaled using the same numbers, but never used for learning.')

---
## Step 8: Hands-on — which scaler wins on a real model? 🎯

Let's train a **Logistic Regression** model to predict `is_late`.

**Logistic Regression** = the classic yes/no model. It uses math that CARES about scaling a lot.

We'll try 4 versions:
- No scaling
- StandardScaler
- MinMaxScaler
- RobustScaler

Which one will give the best accuracy?

In [ ]:
# Test all 4 versions
scalers = {
    'No scaling':     None,
    'StandardScaler': StandardScaler(),
    'MinMaxScaler':   MinMaxScaler(),
    'RobustScaler':   RobustScaler()
}

print('🏁 Training Logistic Regression with each scaler...\n')

results = {}
for name, scaler in scalers.items():
    if scaler is None:
        Xtr, Xte = X_train.values, X_test.values
    else:
        Xtr = scaler.fit_transform(X_train)   # fit on train only!
        Xte = scaler.transform(X_test)         # transform test only
    
    clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    clf.fit(Xtr, y_train)
    acc = accuracy_score(y_test, clf.predict(Xte))
    results[name] = acc
    print(f'   {name:18s} → {acc*100:.2f}% accuracy')

print()
print('━' * 55)
best  = max(results, key=results.get)
worst = min(results, key=results.get)
diff  = (results[best] - results[worst]) * 100
print(f'🏆 BEST : {best} ({results[best]*100:.2f}%)')
print(f'😢 WORST: {worst} ({results[worst]*100:.2f}%)')
print(f'   Difference: {diff:.1f} percentage points')
print()
print('💡 Note: with class_weight="balanced" the gap is small;')
print('   without it, scaling matters a LOT more on logistic regression.')
print('━' * 55)

### 🤔 Stop and think

1. By how much did the BEST scaler beat the WORST?
2. Why do we use `class_weight='balanced'` on Olist `is_late`? (Hint: only 7% of orders are late!)
3. Imagine 10,000 orders and 1% accuracy difference. How many predictions are now correct that were wrong before?

---
## Step 9: Save the scaled data 💾

We will use this scaled data in **Class 3** (feature engineering).

We pick the scaler that worked best (or most popular: StandardScaler) and save the result.

In [ ]:
import joblib

# Re-fit on ALL the data we have, then save (we'll re-split fresh in M4)
scaler = StandardScaler()
scaled = scaler.fit_transform(X)

# Save the scaled features alongside other Olist columns we want to keep
scaled_df = pd.DataFrame(scaled, columns=cols, index=X.index)
out = df.copy()
for c in cols:
    out[c + '_scaled'] = scaled_df[c]

# Save the data + the scaler object
out.to_parquet('orders_step2.parquet', index=False)
joblib.dump(scaler, 'olist_scaler.joblib')

print('✅ Saved 2 files:')
print('   • orders_step2.parquet  → scaled data for Class 3')
print('   • olist_scaler.joblib   → the scaler object (load it next class)')
print()
print('💡 Why save the scaler too?')
print('   When a NEW Olist order arrives in production, we need to scale it')
print('   using the SAME scaler from training. We can NOT re-fit it!')
print()
print('🔗 In M8 (Capstone) — your /predict API will load olist_scaler.joblib')
print('   and use it on every incoming order. This file is gold.')

---
## 🏁 We did it!

### Today's summary

| Topic | What we learned | Tool |
| --- | --- | --- |
| Why scale? | Different sizes confuse the model | — |
| Tool 1 | Mean=0, Std=1. Best default. | `StandardScaler` |
| Tool 2 | Squeeze to [0, 1]. Risky with outliers. | `MinMaxScaler` |
| Tool 3 | Uses median + IQR. Best with outliers. | `RobustScaler` |
| The big rule | Fit on TRAIN only. Transform TEST. | `train_test_split` |
| Real test | Logistic Regression with each scaler on `is_late` | `accuracy_score` |

### 🎓 Words to remember

- **scaling** — make all numbers fit in a similar range
- **standardization** — mean=0, std=1 (StandardScaler)
- **normalization** — squeeze to [0, 1] (MinMaxScaler)
- **mean** — average
- **std** — standard deviation = how spread out
- **median** — middle value
- **IQR** — middle 50% range (Q3 − Q1)
- **train/test split** — split data into learning set + check set
- **data leakage** — secret info from test sneaks into train (bad!)
- **outlier** — a value far from the rest
- **accuracy** — % of correct predictions

### 🗺️ Where you'll see scaling again in this course

| Module | Class | Model | Needs scaling? |
| --- | --- | --- | --- |
| **M4** | Class 1 | Linear Regression (predict freight from distance on Olist) | ✅ YES |
| **M4** | Class 2 | Logistic Regression (predict `is_late`) | ✅ YES |
| **M4** | Class 3 | Decision Trees + Random Forest | ❌ No |
| **M4** | Class 4 | Model Evaluation | (no model) |
| **M5** | Class 2 | K-Means RFM customer segmentation | ✅ YES (distance!) |
| **M6** | All | Neural Networks on Olist | ✅ YES |
| **M8** | Class 3 | Deployed FastAPI — loads `olist_scaler.joblib` to scale new incoming orders | ✅ YES |

**Today's lesson is the foundation for the next 6 weeks of teaching!**

### 🎯 Where we're going next

- ✅ Class 1: Clean data
- ✅ Class 2: Scale numbers (today!)
- 📅 Class 3: **Feature engineering** — build NEW useful columns like `distance_km`, `is_weekend`, `delivery_days`
- 📅 Class 4: Feature selection — pick the best ones
- 📅 Class 5: Pipelines — automate everything
- 📅 Class 6: **The big lab** — produce `olist_clean.parquet` end to end
- 📅 Module 4: Train the model. Predict which orders will be late!

### 📤 Submit

1. Save as `Module3_Class2_<YourName>.ipynb`
2. PR to your group repo at `module-3/class_2/submissions/<YourName>/`
3. Include `orders_step2.parquet` and `olist_scaler.joblib`

📐 *Great work! See you in Class 3 — feature engineering!*